In [ ]:
## import dependencies
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import pandas as pd
import torch
import plotly
import requests
#import STalign
#from STalign import STalign
import pickle
import json
import cv2
import scanpy as sc
import anndata as ad

# make plots bigger
plt.rcParams["figure.figsize"] = (12,10)

### Visium data

In [ ]:
he_id = 'MSi_V11L12-109_B1'
adata = sc.read_h5ad(f'../{he_id}_visium_data.h5ad')

In [ ]:
adata.var_names_make_unique()

In [ ]:
adata

In [ ]:
adata.obs['x'] = np.array(adata.obsm['spatial'][:, 0])
adata.obs['y'] = np.array(adata.obsm['spatial'][:, 1])
adata.layers['counts'] = adata.X.copy()

In [ ]:
with open(f'../MSi_V11L12-038_D1_spatialDE_genes.pickle', 'rb') as handle:
    gene_list = pickle.load(handle)

In [ ]:
np.sum([gene in adata.var_names for gene in gene_list])

In [ ]:
adata = adata[:, gene_list].copy()

In [ ]:
import sys
import os

# Get the absolute path to the directory containing the script to be imported
script_dir = os.path.abspath("../SuperFocus")

# Add the directory to sys.path
sys.path.append(script_dir)

from importlib import reload
import src.model_training.model_training_orig
import src.model_training.processing_image_features
reload(src.model_training.model_training_orig)
reload(src.model_training.processing_image_features)
from src.model_training.model_training_orig import training_orig

In [ ]:
sample_id = he_id + '_visium'

In [ ]:
sample_id

In [ ]:
from catalyst.utils import set_global_seed
set_global_seed(123)
training_orig(adata=adata, pt_model_dir=f'../CellViT/maldi_mousebrain/preprocessing/{he_id}/{he_id}/cell_detection', radius=376./2, feature_list=list(gene_list), 
              save_model_dir=f'./logs_{sample_id}', hidden_dims=[8192, 4096, 4096, 8192], activations=None, residual=False, norm=False, out_act=None, 
              include_tanhshrink=False, sample_id=sample_id, training_mode=None, selected_model='lite_shrink', kfold=3, cell_embed_dir=f'./{sample_id}_cell_embed_df.pickle', datatype='Visium')

In [ ]:
import src.post_process
from importlib import reload
reload(src.post_process)
from src.post_process import post_training_processing

In [ ]:
adata_pred = post_training_processing.post_procesing(model_dir=f'./logs_{sample_id}/checkpoints/model.best.pth', hidden_dims=[8192, 4096, 4096, 8192], 
                                               activations=None, out_act=None, residual=False, norm=False, selected_model='lite_shrink', include_tanhshrink=False, 
                                               data_dir=f'./{sample_id}_training_data_level1.pickle', feature_list=list(gene_list), datatype='RNA', regress_out=False, resolutions=[0.6, 0.8],
                                               refine_factor=1, sample_id=sample_id, radius=376./2)

In [ ]:
adata_pred

In [ ]:
sc.pl.spatial(adata_pred, img_key=None, color='leiden_res_0.6')

In [ ]:
import src.model_training.model_training_refined
reload(src.model_training.model_training_refined)
from src.model_training.model_training_refined import training

In [ ]:
from catalyst.utils import set_global_seed
set_global_seed(123)
training(adata=adata, pt_model_dir=f'../CellViT/maldi_mousebrain/preprocessing/{he_id}/{he_id}/cell_detection', radius=376./2, refine_factor=2, 
         feature_pd_dir=f'./{sample_id}_gene_exp_pd.pickle', 
         trained_model_dir=f'./logs_{sample_id}/checkpoints/model.best.pth',
         save_model_dir=f'./logs_{sample_id}_2x', hidden_dims=[8192, 4096, 4096, 8192], activations=None, residual=False, norm=False, out_act=None, 
         include_tanhshrink=False, sample_id=sample_id, training_mode=None, selected_model='lite_shrink', kfold=3, cell_embed_dir=f'./{sample_id}_cell_embed_df.pickle', 
         lr=1e-7, step_size=25, gamma=0.5, num_epochs=50, random_state=42, datatype='Visium')

In [ ]:
adata_pred_level2 = post_training_processing.post_procesing(model_dir=f'./logs_{sample_id}_2x/best_model_fold1.pth', hidden_dims=[8192, 4096, 4096, 8192], 
                                               activations=None, out_act=None, residual=False, norm=False, selected_model='lite_shrink', include_tanhshrink=False, 
                                               data_dir=f'./{sample_id}_training_data_level2.pickle', feature_list=list(gene_list), datatype='RNA', regress_out=False, resolutions=[0.4, 0.6],
                                               refine_factor=2, sample_id=sample_id, radius=376./2)

In [ ]:
sc.pl.spatial(adata_pred_level2, img_key=None, color='leiden_res_0.6')

In [ ]:
training(adata=adata, pt_model_dir=f'../CellViT/maldi_mousebrain/preprocessing/{he_id}/{he_id}/cell_detection', radius=376./2, refine_factor=4, 
         feature_pd_dir=f'./{sample_id}_gene_exp_pd.pickle', 
         trained_model_dir=f'./logs_{sample_id}_2x/best_model_fold1.pth',
         save_model_dir=f'./logs_{sample_id}_4x', hidden_dims=[8192, 4096, 4096, 8192], activations=None, residual=False, norm=False, out_act=None, 
         include_tanhshrink=False, sample_id=sample_id, training_mode=None, selected_model='lite_shrink', kfold=5, cell_embed_dir=f'./{sample_id}_cell_embed_df.pickle', 
         lr=1e-7, step_size=25, gamma=0.5, num_epochs=50, random_state=42, datatype='Visium')

In [ ]:
adata_pred_level4 = post_training_processing.post_procesing(model_dir=f'./logs_{sample_id}_4x/best_model_fold1.pth', hidden_dims=[8192, 4096, 4096, 8192], 
                                               activations=None, out_act=None, residual=False, norm=False, selected_model='lite_shrink', include_tanhshrink=False, 
                                               data_dir=f'./{sample_id}_training_data_level4.pickle', feature_list=list(gene_list), datatype='RNA', regress_out=False, resolutions=[0.4, 0.6],
                                               refine_factor=4, sample_id=sample_id, radius=376./2)

In [ ]:
sc.pl.spatial(adata_pred_level4, img_key=None, color='leiden_res_0.6')

### apply to the whole tissue slide

In [ ]:
from src.model_training.processing_image_features import whole_image_all_cells_features
from src.model_training import MLP

In [ ]:
refine_factor = 4
cell_markers, cell_counts, cell_coords = whole_image_all_cells_features(pt_model_dir=f'../../CellViT/Visium/preprocessing/{he_id}/{he_id}/cell_detection', radius=378./2/refine_factor, cell_embed_dir=f'./{sample_id}_cell_embed_df.pickle')
cellvit_marker_array = np.hstack((cell_markers, cell_counts))

In [ ]:
with open(f'./{sample_id}_scaler_refine_level{refine_factor}.pickle', 'rb') as file:
    CODEX_HIST_scaler = pickle.load(file)
    
with open(f'./{sample_id}_gene_exp_pd.pickle', 'rb') as file:
    dbit_pd = pickle.load(file)

In [ ]:
codex_cell_marker_scaled = CODEX_HIST_scaler.transform(cellvit_marker_array)

In [ ]:
hidden_dims=[8192, 4096, 4096, 8192]
activations=None
residual=False
norm=False
out_act=None
model = MLP.Model_lite_shrink(n_inp=3609, n_out=3000, hidden_dims=hidden_dims, activations=activations, out_activation=out_act)

state_dict = torch.load(f'./logs_{sample_id}_4x/best_model_fold1.pth', map_location=torch.device('cpu')) # Load on CPU
model.load_state_dict(state_dict)

model.eval()

with torch.no_grad():
    output_full = model(torch.tensor(codex_cell_marker_scaled).to(torch.float32))

In [ ]:
predicted_gene_full= np.array(output_full) * cell_counts[:, 0][:, None]
predicted_gene_full = np.maximum(0, predicted_gene_full)

## Create adata object
adata = ad.AnnData(predicted_gene_full)#ad.AnnData(sparse.csc_matrix(predicted_gene))

#adata.obs_names = list(dbit_pd.index)
adata.var_names = list(dbit_pd.columns)

adata.obs['x'] = np.array(cell_coords['pxl_col_in_fullres'])
adata.obs['y'] = np.array(cell_coords['pxl_row_in_fullres'])
adata.obs['section_id'] = [he_id]*adata.n_obs

adata.layers['counts'] = predicted_gene_full.copy()

In [ ]:
adata.uns = {}
adata.uns['spatial'] = {}
adata.uns['spatial'][sample_id] = {}
adata.uns['spatial'][sample_id]['images'] = {}
#adata_query.uns['spatial'][sample_id]['images']['lowres'] = imageLN_flip # rotated_imageLN
adata.uns['spatial'][sample_id]['scalefactors'] = {}
adata.uns['spatial'][sample_id]['scalefactors']['tissue_hires_scalef'] = 16.0  # 78.5
adata.uns['spatial'][sample_id]['scalefactors']['spot_diameter_fullres'] =  20.0 #2.18
adata.uns['spatial'][sample_id]['scalefactors']['fiducial_diameter_fullres'] = 4.24
adata.uns['spatial'][sample_id]['scalefactors']['tissue_lowres_scalef'] = 16.0 # 78.5

adata.obsm['spatial'] = np.array([adata.obs['x'], adata.obs['y']]).T
adata.obs['sample_id'] = [he_id] * adata.n_obs

In [ ]:
adata.X = adata.layers['counts'].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
#sc.pp.highly_variable_genes(adata, n_top_genes=500)

# scale data, clip values exceeding standard deviation 10.
sc.pp.scale(adata, max_value=10)

sc.tl.pca(adata, svd_solver='arpack')
# plot more PCS
sc.pl.pca(adata, components = ['1,2','3,4','5,6','7,8'], ncols=2)

In [ ]:
# Then we calculate a new umap
sc.pp.neighbors(adata, n_neighbors=50, n_pcs=25, use_rep='X_pca', metric='cosine', random_state=0)

sc.tl.umap(adata, min_dist=0.0, random_state=0)

In [ ]:
sc.tl.leiden(adata, resolution = 0.4, key_added = "leiden_0.4", random_state=0, flavor='igraph', n_iterations=2)
sc.tl.leiden(adata, resolution = 0.5, key_added = "leiden_0.5", random_state=0, flavor='igraph', n_iterations=2)
sc.tl.leiden(adata, resolution = 0.6, key_added = "leiden_0.6", random_state=0, flavor='igraph', n_iterations=2)
sc.tl.leiden(adata, resolution = 0.7, key_added = "leiden_0.7", random_state=0, flavor='igraph', n_iterations=2)
sc.tl.leiden(adata, resolution = 0.8, key_added = "leiden_0.8", random_state=0, flavor='igraph', n_iterations=2)

In [ ]:
sc.pl.umap(adata, color='leiden_0.6')
sc.pl.umap(adata, color='leiden_0.7')
sc.pl.umap(adata, color='leiden_0.8')

In [ ]:
sc.pl.spatial(adata, color='leiden_0.4', img_key=None, size=4.0)

In [ ]:
sc.pl.spatial(adata, color='leiden_0.5', img_key=None, size=4.0)

In [ ]:
adata.write_h5ad(f'{sample_id}_full_adata.h5ad')